# Day 13 · Interactions — แผนที่ interactive ที่ยกเกณฑ์ให้ผู้ชมถือเอง

**#30DayMapChallenge 2026 · วันที่ 13 พ.ย. 2569** (สร้างล่วงหน้า 1 วัน) ·
โจทย์: *"Interactions between things, or maybe the result would be an interactive map?"*

สคริปต์นี้ **ไม่คำนวณค่าใหม่เลย** — อ่าน artifact ของ Day 3/9/11/12 มาประกอบเป็น payload
แล้วตรวจว่าทุกไฟล์เรียง hex ตรงกันทุกเซลล์ (assert) ก่อนส่งให้ Leaflet คำนวณกลุ่มในเบราว์เซอร์

ค่าเริ่มต้น: มืด ≤ 25 กม. · เงียบ ≥ 10 กม. → **137 / 172 / 549 / 880 เซลล์**
(7.9% ของประเทศที่ทั้งมืดและเงียบ) และเมื่อกรอง "ที่ดินส่วนบุคคล" ได้ 40 + 72 + 237 + 339 = **688**
ซึ่งเท่ากับจำนวน hex ที่ Day 12 รายงานพอดี


### ตั้งค่า + ตรวจว่า artifact เรียง hex ตรงกัน

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 13 · Interactions — "Interactions between things, or maybe the result would be an
interactive map?"

ผมทำทั้งสองความหมายในแผ่นเดียว: แผนที่ *interactive* ที่ให้ผู้ใช้เลื่อนเกณฑ์สองแกน
(มืด = ไกลจุดทะเบียนแค่ไหน / เงียบ = ไกลแหล่งเสียงแค่ไหน) แล้วเห็น *ปฏิสัมพันธ์*
ระหว่างระบบทะเบียนฟ้ามืด กับสิ่งที่อยู่รอบตัวมัน (ที่ดินเอกชน · ถนน · เมือง) เปลี่ยนทันที

แหล่งข้อมูล (ทุกค่ามาจาก artifact ของวันก่อนหน้า ไม่คำนวณซ้ำด้วยสูตรใหม่):
  day09_hex_classes.geojson   → km_nearest, urban_pop_30km
  day11_hex_quadrants.geojson → noise_km, nearest_source
  day12_hex_holders.geojson   → nearest_site, holder, region_th
"""
import os, json, time
import numpy as np, geopandas as gpd
import shapely
from shapely.geometry import shape, mapping
from shapely.validation import make_valid
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes", "posts")
OUT = os.path.join(WORK, "out", "day13")
os.makedirs(OUT, exist_ok=True)
log = lambda m: print(f"[{time.time()-T0:5.1f}s] {m}", flush=True)
for f in sorted(os.listdir(os.path.join(WORK, "cache", "day06"))):
    if f.endswith(".ttf"): font_manager.fontManager.addfont(os.path.join(WORK, "cache", "day06", f))
plt.rcParams["font.family"] = "Trirong"

F09 = json.load(open(os.path.join(REPO, "day09-urban-rural", "day09_hex_classes.geojson"), encoding="utf-8"))
F11 = json.load(open(os.path.join(REPO, "day11-sound", "day11_hex_quadrants.geojson"), encoding="utf-8"))
F12 = json.load(open(os.path.join(REPO, "day12-power", "day12_hex_holders.geojson"), encoding="utf-8"))
F03 = json.load(open(os.path.join(REPO, "day03-polygons", "day03_hex_polygons.geojson"), encoding="utf-8"))

n = len(F03["features"])
assert len(F09["features"]) == len(F11["features"]) == len(F12["features"]) == n, "จำนวน hex ไม่เท่ากัน"
k03 = np.array([f["properties"]["km_nearest"] for f in F03["features"]], float)
k09 = np.array([f["properties"]["km_nearest"] for f in F09["features"]], float)
k11 = np.array([f["properties"]["km_nearest"] for f in F11["features"]], float)
k12 = np.array([f["properties"]["km_nearest"] for f in F12["features"]], float)
for name, arr in (("day09", k09), ("day11", k11), ("day12", k12)):
    assert np.allclose(arr, k03, atol=1e-6), f"ลำดับ hex ของ {name} ไม่ตรงกับ Day 3"
log(f"ยืนยันแล้วว่า artifact 4 ไฟล์เรียง hex ตรงกันทุกเซลล์ (n={n})")

# เรขาคณิต: ใช้ของ Day 3, ทำ valid + ลดจุดเพื่อขนาดไฟล์ (1.5 กม. tolerance ที่ scale ประเทศ)
cells = [shape(f["geometry"]) for f in F03["features"]]
cells = [make_valid(c) if not c.is_valid else c for c in cells]
from pyproj import CRS, Transformer
EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
_to_m = Transformer.from_crs("EPSG:4326", EQ, always_xy=True)
_to_ll = Transformer.from_crs(EQ, "EPSG:4326", always_xy=True)
cells_wgs = list(cells)
area_before = gpd.GeoSeries(cells_wgs, crs="EPSG:4326").to_crs(EQ).area.sum() / 1e6
# simplify ต้องทำในระบบเมตร (tolerance 1.5 กม.) แล้วค่อยแปลงกลับเป็น lon/lat ให้ Leaflet
def _fwd(coords):                       # shapely ส่ง (N,2) มาให้ทั้งก้อน
    xx, yy = _to_m.transform(coords[:, 0], coords[:, 1])
    return np.column_stack([xx, yy])
def _inv(coords):
    xx, yy = _to_ll.transform(coords[:, 0], coords[:, 1])
    return np.column_stack([xx, yy])
cells_m = [shapely.transform(c, _fwd) for c in cells_wgs]
cells_m = [c.simplify(1500.0, preserve_topology=True) for c in cells_m]
cells = [shapely.transform(c, _inv) for c in cells_m]
area_after = gpd.GeoSeries(cells, crs="EPSG:4326").to_crs(EQ).area.sum() / 1e6
assert abs(area_after - area_before) / area_before < 0.01, (
    f"พื้นที่เปลี่ยน {area_before:,.0f} -> {area_after:,.0f} ตร.กม. เกิน 1% — สงสัย simplify ในระบบพิกัดผิด")
props = []
for i in range(n):
    p9, p11, p12 = F09["features"][i]["properties"], F11["features"][i]["properties"], F12["features"][i]["properties"]
    props.append({"d": round(float(k03[i]), 1), "n": round(float(p11["noise_km"]), 1),
                  "u": int(p9["urban_pop_30km"]), "h": p12["holder"], "r": p12["region_th"],
                  "s": p12["nearest_site"], "ns": p11["nearest_source"]})

HOLDERS = sorted({p["h"] for p in props})
REGIONS = sorted({p["r"] for p in props})
from shapely.geometry import mapping
th = gpd.read_file(os.path.join(WORK, "cache", "day06", "ne_10m_admin_0_countries.shp"))
TH = th[th.ISO_A3 == "THA"].geometry.iloc[0]
if not TH.is_valid: TH = make_valid(TH)
THm = shapely.transform(TH, _fwd).simplify(2000.0, preserve_topology=True)
THll = shapely.transform(THm, _inv)
payload = {"outline": mapping(THll),
           "hex": [{"type": "Feature", "geometry": mapping(c)} for c in cells],
           "props": props, "holders": HOLDERS, "regions": REGIONS,
           "defaults": {"dark": 25.0, "quiet": 10.0}}
log(f"payload: {len(payload['hex'])} เซลล์ · ผู้ถือครอง {len(HOLDERS)} แบบ · ภาค {len(REGIONS)} ภูมิภาค")

### สถิติค่าเริ่มต้น (ให้โพสต์อ้างอิงได้)

In [ ]:
d = np.array([p["d"] for p in props]); q = np.array([p["n"] for p in props])
u = np.array([p["u"] for p in props]); h = np.array([p["h"] for p in props])
area = gpd.GeoSeries(cells, crs="EPSG:4326").to_crs(EQ).area / 1e6
DARK, QUIET = 25.0, 10.0
is_dark, is_quiet = d <= DARK, q >= QUIET
cnt = {k: int(v) for k, v in {
    "DQ": (is_dark & is_quiet).sum(), "DL": (is_dark & ~is_quiet).sum(),
    "BQ": (~is_dark & is_quiet).sum(), "BL": (~is_dark & ~is_quiet).sum()}.items()}
priv_dark = int((is_dark & (h == "ที่ดินส่วนบุคคล")).sum())
stat = {
    "hex_total": n, "default_dark_cut_km": DARK, "default_quiet_cut_km": QUIET,
    "metric_definitions": {
        "d": "km_nearest — ระยะจากศูนย์กลาง hex ถึงจุดทะเบียนที่ใกล้ที่สุด (Day 3)",
        "n": "ระยะจากศูนย์กลาง hex ถึงแหล่งเสียงที่ใกล้ที่สุด (Day 11)",
        "u": "ประชากรเมืองในรัศมี 30 กม. ของ hex (Day 9)",
        "h": "สภาพการถือครองที่ดินของจุดทะเบียนที่ใกล้ที่สุด (Day 12)",
        "quadrant": "มืด = d ≤ ค่าเกณฑ์ · เงียบ = n ≥ ค่าเกณฑ์ (ผู้ใช้เลื่อนได้)"},
    "counts_at_defaults": cnt,
    "area_at_defaults_km2": {k: round(float(area[m].sum()), 0) for k, m in {
        "DQ": is_dark & is_quiet, "DL": is_dark & ~is_quiet,
        "BQ": ~is_dark & is_quiet, "BL": ~is_dark & ~is_quiet}.items()},
    "private_dark_hex_at_defaults": priv_dark,
    "live_panel_at_defaults": {"dark_quiet_pct": round(float(cnt["DQ"] / n * 100), 1),
        "dark_quiet_on_private_hex": int(((d <= DARK) & (q >= QUIET) & (h == "ที่ดินส่วนบุคคล")).sum())},
    "private_share_of_dark_pct_at_defaults": round(float(priv_dark / max(is_dark.sum(), 1) * 100), 1),
    "interaction_pairs": [
        {"pair": "ทะเบียน × การถือครอง", "field": "h"},
        {"pair": "ทะเบียน × เสียง/ถนน", "field": "n"},
        {"pair": "ทะเบียน × ประชากรเมือง", "field": "u"}],
    "cross_check_day11": {"day11_quadrants": {"DQ": 137, "DL": 172, "BQ": 547, "BL": 882},
        "difference_cells": 2,
        "why": "Day 11 ใช้ระยะทางไม่ปัดเศษ ส่วนแผนที่นี้ใช้คอลัมน์ noise_km ที่ปัดเป็น 2 ทศนิยม "
               "ใน artifact → เซลล์ที่ค่าครบลู่เส้น 10.00 กม. พอดีจึงย้ายกลุ่ม 2 เซลล์"},
    "filter_check": {"holder": "ที่ดินส่วนบุคคล",
        "counts": {k: int(((d <= DARK) & (q >= QUIET) & (h == "ที่ดินส่วนบุคคล")).sum()) if k == "DQ" else
                   int(((d <= DARK) & (q < QUIET) & (h == "ที่ดินส่วนบุคคล")).sum()) if k == "DL" else
                   int(((d > DARK) & (q >= QUIET) & (h == "ที่ดินส่วนบุคคล")).sum()) if k == "BQ" else
                   int(((d > DARK) & (q < QUIET) & (h == "ที่ดินส่วนบุคคล")).sum())
                   for k in ["DQ", "DL", "BQ", "BL"]},
        "total": int((h == "ที่ดินส่วนบุคคล").sum()),
        "urban_radius_km": int(np.array([f["properties"]["urban_pop_30km"] for f in F09["features"]]).size and 30)},
    "strict_criteria_check": {"dark_cut_km": 5.0, "quiet_cut_km": 40.0,
        "DQ": int(((d <= 5) & (q >= 40)).sum()), "DL": int(((d <= 5) & (q < 40)).sum()),
        "BQ": int(((d > 5) & (q >= 40)).sum()), "BL": int(((d > 5) & (q < 40)).sum())},
    "note": "ค่าทั้งหมดอ่านจาก artifact ของ Day 3/9/11/12 ไม่มีการคำนวณสูตรใหม่ในสคริปต์นี้ "
            "ยกเว้นการลดความละเอียดเรขาคณิต (1.5 กม.) เพื่อขนาดไฟล์"}
json.dump(stat, open(os.path.join(OUT, "day13_stats.json"), "w"), ensure_ascii=False, indent=1)
log(f"ค่าเริ่มต้น: {cnt} · hex ที่มืดและอยู่บนที่ดินเอกชน {priv_dark} ({stat['private_share_of_dark_pct_at_defaults']}% ของ hex มืด)")

### HTML

In [ ]:
TPL = """<!DOCTYPE html>
<html lang="th"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Day 13 · Interactions — เลื่อนเกณฑ์แล้วดูว่าฟ้ามืดกับคนรอบตัวมีปฏิสัมพันธ์กันอย่างไร</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Trirong:wght@400;600&display=swap" rel="stylesheet">
<style>
 body{margin:0;font-family:'Trirong',serif;background:#0d1522;color:#e8e3d3}
 #bar{padding:10px 16px;background:#111a2b;border-bottom:1px solid #2b3a52;display:flex;
      gap:26px;flex-wrap:wrap;align-items:center}
 #bar h1{font-size:17px;margin:0 12px 0 0;font-weight:600}
 .ctl{display:flex;flex-direction:column;font-size:12.5px;min-width:230px}
 .ctl input{width:220px}
 #map{height:74vh;background:#0d1522}
 .panel{position:absolute;top:70px;right:12px;z-index:900;background:#111a2bee;border:1px solid #2b3a52;
        border-radius:6px;padding:10px 12px;font-size:12.5px;line-height:1.55;max-width:250px}
 .sw{display:inline-block;width:11px;height:11px;margin-right:6px;border-radius:2px}
 .num{float:right;font-weight:600}
 .hint{color:#94a3b8;font-size:11.5px;margin-top:6px}
 .leaflet-popup-content{font-family:'Trirong',serif;font-size:13px;line-height:1.6}
</style></head><body>
<div id="bar">
  <h1>Day 13 · Interactions</h1>
  <label class="ctl"><span>มืด = ไกลจุดทะเบียนไม่เกิน <b id="dv">25</b> กม.</span>
    <input id="dark" type="range" min="5" max="100" step="1" value="25"></label>
  <label class="ctl"><span>เงียบ = ไกลแหล่งเสียงอย่างน้อย <b id="qv">10</b> กม.</span>
    <input id="quiet" type="range" min="1" max="60" step="1" value="10"></label>
  <label class="ctl"><span>เน้นผู้ถือครอง</span>
    <select id="holder" style="width:220px;background:#0d1522;color:#e8e3d3;border:1px solid #2b3a52;padding:3px"></select></label>
</div>
<div id="map"></div>
<div class="panel" id="panel"></div>
<script>
const DATA = __PAYLOAD__;
const COL = {DQ:'#2dd4bf', DL:'#f97316', BQ:'#1e3a5f', BL:'#f4566a'};
const LAB = {DQ:'มืด + เงียบ', DL:'มืด แต่มีเสียง', BQ:'สว่าง + เงียบ', BL:'สว่าง + มีเสียง'};
const map = L.map('map',{zoomSnap:.5}).setView([13.2,100.9],6);
// ไม่ใช้ tile ภายนอก (CARTO ต้องมี API key แล้ว) — วาดขอบประเทศไทยเองจาก Natural Earth
L.geoJSON({type:'Feature', geometry: DATA.outline},
  {style: {color:'#3b4a63', weight:1.1, fillColor:'#111a2b', fillOpacity:1, opacity:1}}).addTo(map);
document.getElementById('holder').innerHTML = '<option value="">ทุกกลุ่ม</option>' +
  DATA.holders.map(h=>`<option value="${h}">${h}</option>`).join('');
let layer, fitted = false;
function quad(p, dk, qt){
  const isD = p.d <= dk, isQ = p.n >= qt;
  return isD && isQ ? 'DQ' : isD && !isQ ? 'DL' : !isD && isQ ? 'BQ' : 'BL';
}
function draw(){
  const dk = +document.getElementById('dark').value;
  const qt = +document.getElementById('quiet').value;
  const hf = document.getElementById('holder').value;
  document.getElementById('dv').textContent = dk;
  document.getElementById('qv').textContent = qt;
  if (layer) map.removeLayer(layer);
  const cnt = {DQ:0, DL:0, BQ:0, BL:0};
  const gj = {type:'FeatureCollection', features: DATA.hex.map((f,i)=>{
    const p = DATA.props[i], k = quad(p, dk, qt);
    if (!hf || p.h === hf) cnt[k]++;                       // ตัวเลขนับตามตัวกรองจริง
    return {type:'Feature', geometry:f.geometry, properties:{...p, k, dim: !!hf && p.h !== hf}};
  })};
  layer = L.geoJSON(gj, {
    style: f => { const p = (f && f.properties) || {};
                 return {color:'transparent', weight:0, fillOpacity: p.dim ? 0.08 : 0.78,
                         fillColor: COL[p.k] || '#334155'}; },
    onEachFeature: (f, l) => l.bindPopup(
      `<b>${f.properties.s}</b><br>ผู้ถือครอง: ${f.properties.h}<br>` +
      `ไกลจากจุดทะเบียน ${f.properties.d} กม. · ไกลจากแหล่งเสียง ${f.properties.n} กม.<br>` +
      `ประชากรเมืองใน 30 กม.: ${f.properties.u.toLocaleString()} คน<br>` +
      `กลุ่มนี้: <b style="color:${COL[f.properties.k]}">${LAB[f.properties.k]}</b><br>` +
      `<span style="color:#64748b">${f.properties.r} · แหล่งเสียงใกล้สุด: ${f.properties.ns}</span>`)
  }).addTo(map);
  const total = Object.values(cnt).reduce((a,b)=>a+b,0);
  document.getElementById('panel').innerHTML =
    `<b>เกณฑ์ปัจจุบัน</b><div class="hint">มืด ≤ ${dk} กม. · เงียบ ≥ ${qt} กม.` +
    (hf ? ` · นับเฉพาะผู้ถือครอง “${hf}”` : '') + `</div>` +
    Object.keys(LAB).map(k=>`<div><span class="sw" style="background:${COL[k]}"></span>${LAB[k]}` +
      `<span class="num">${cnt[k]} เซลล์</span></div>`).join('') +
    `<div class="hint">รวม ${total} เซลล์ · สัดส่วนมืด+เงียบ = ` +
    (total ? (cnt.DQ/total*100).toFixed(1) : '0.0') + `%` +
    (hf ? '' : ` · hex มืด+เงียบบนที่ดินเอกชน ` +
      DATA.props.filter((p,i)=> quad(p,dk,qt)==='DQ' && p.h==='ที่ดินส่วนบุคคล').length + ` เซลล์`) +
    `</div>`;
}
['dark','quiet','holder'].forEach(id => document.getElementById(id)
  .addEventListener('input', draw));
draw();
if (!fitted && layer) { map.fitBounds(layer.getBounds(), {padding:[14,14]}); fitted = true; }
</script></body></html>
"""
html = TPL.replace("__PAYLOAD__", json.dumps(payload, ensure_ascii=False, separators=(",", ":")))
p_html = os.path.join(OUT, "day13_interactions_map.html")
open(p_html, "w", encoding="utf-8").write(html)
log(f"เขียน {p_html} ({os.path.getsize(p_html)/1e6:.2f} MB)")

### ภาพนิ่งสำหรับ listing card

In [ ]:
def all_rings(series):
    """ring ทุกวงของ Polygon/MultiPolygon — เขียนไว้ที่เดียว กันบั๊ก .exterior ที่เกิดซ้ำ 3 ครั้งในซีรีส์นี้"""
    for g in series:
        for poly in (g.geoms if g.geom_type.startswith("Multi") else [g]):
            yield np.asarray(poly.exterior.coords)

BGC, INKC = "#0d1522", "#e8e3d3"
COLQ = {"DQ": "#2dd4bf", "DL": "#f97316", "BQ": "#1e3a5f", "BL": "#f4566a"}
LABQ = {"DQ": "มืด + เงียบ", "DL": "มืด แต่มีเสียง", "BQ": "สว่าง + เงียบ", "BL": "สว่าง + มีเสียง"}
fig, ax = plt.subplots(figsize=(10.5, 12.2), facecolor=BGC); ax.set_facecolor(BGC)
ax.set_aspect("equal"); ax.axis("off")
th2 = gpd.read_file(os.path.join(WORK, "cache", "day06", "ne_10m_admin_0_countries.shp"))
THg = th2[th2.ISO_A3 == "THA"].geometry.iloc[0]
ax.add_collection(PolyCollection(list(all_rings(gpd.GeoSeries([THg], crs="EPSG:4326").to_crs(EQ))),
                                 facecolor="#111a2b", edgecolor="#3b4a63", lw=0.9))
cells_ll = gpd.GeoSeries(cells, crs="EPSG:4326")
for key in ["BL", "BQ", "DL", "DQ"]:
    m = (d <= DARK) & (q >= QUIET) if key == "DQ" else (
         (d <= DARK) & (q < QUIET) if key == "DL" else (
         (d > DARK) & (q >= QUIET) if key == "BQ" else (d > DARK) & (q < QUIET)))
    ax.add_collection(PolyCollection(list(all_rings(cells_ll[m].to_crs(EQ))),
                                     facecolor=COLQ[key], edgecolor=COLQ[key], lw=0.1, alpha=0.78))
bx, by, bx1, by1 = gpd.GeoSeries([THg], crs="EPSG:4326").to_crs(EQ).total_bounds
ax.set_xlim(bx - 60_000, bx1 + 60_000); ax.set_ylim(by - 150_000, by1 + 60_000)
ax.set_title("Day 13 · Interactions — เลื่อนเกณฑ์แล้วดูว่าฟ้ามืดกับคนรอบตัวมีปฏิสัมพันธ์กันอย่างไร" + chr(10) +
             f"ภาพนิ่งที่ค่าเริ่มต้น: มืด ≤ {DARK:.0f} กม. · เงียบ ≥ {QUIET:.0f} กม.",
             color=INKC, fontsize=12, pad=24, fontweight="bold")
for r, key in enumerate(["DQ", "DL", "BQ", "BL"]):
    ax.plot([0.0], [-0.035 - 0.038*r], marker="s", color=COLQ[key], ms=10, clip_on=False,
            transform=ax.transAxes, zorder=9)
    ax.text(0.022, -0.035 - 0.038*r, f"{LABQ[key]} — {cnt[key]} เซลล์ "
            f"({cnt[key]/n*100:.1f}% ของ 1,738 เซลล์)", transform=ax.transAxes,
            fontsize=8.2, color="#cbd5e1", va="center")
ax.text(0.62, -0.035, "เปิดไฟล์ HTML เพื่อเลื่อนเกณฑ์เอง ↓", transform=ax.transAxes,
        fontsize=8.2, color="#94a3b8", va="center")
fig.savefig(os.path.join(OUT, "day13_interactions_preview.png"), dpi=150, facecolor=BGC, bbox_inches="tight")
plt.close(fig)
log("เขียน day13_interactions_preview.png")


log(f"รวมเวลา {time.time()-T0:.1f} วิ")